# Experiment 3 - CDC BRFSS Feature Importance Analysis

This experiment tunes LightGBM and XGBoost on the CDC BRFSS dataset and compares their feature importance rankings.

The goal is to identify which features the tuned models rely on most when predicting diabetes.

Models:
- LightGBM
- XGBoost

Main outputs:
- Tuned model performance
- Top feature importance plots
- Feature importance comparison table

In [1]:
#1 Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from IPython.display import display

print("imports successfully loaded")

imports successfully loaded


In [4]:
#2 Clone repo / move into project folder
if not os.path.exists("/content/Predicting-Type-2-Diabetes"):
    !git clone https://github.com/COMP3608-Intelligent-Systems-Project/Predicting-Type-2-Diabetes.git

%cd /content/Predicting-Type-2-Diabetes

/content/Predicting-Type-2-Diabetes


In [5]:
#3 Load CDC BRFSS data
path = "data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"
df = pd.read_csv(path)

X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

print("-" * 50)
print(" Experiment 3: CDC Feature Importance Analysis")
print("-" * 50)
print(f"Dataset shape: {df.shape}")
print(f"Features shape: {X.shape}")
print("\nTarget balance (%):")
print((y.value_counts(normalize=True) * 100).round(2))

df.head()

--------------------------------------------------
 Experiment 3: CDC Feature Importance Analysis
--------------------------------------------------
Dataset shape: (253680, 22)
Features shape: (253680, 21)

Target balance (%):
Diabetes_binary
0.0    86.07
1.0    13.93
Name: proportion, dtype: float64


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [6]:
#4 Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Train shape: (202944, 21)
Test shape: (50736, 21)

Train target distribution:
Diabetes_binary
0.0    174667
1.0     28277
Name: count, dtype: int64

Test target distribution:
Diabetes_binary
0.0    43667
1.0     7069
Name: count, dtype: int64


In [7]:
#5 Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Before SMOTE:
Diabetes_binary
0.0    174667
1.0     28277
Name: count, dtype: int64

After SMOTE:
Diabetes_binary
0.0    174667
1.0    174667
Name: count, dtype: int64
